In [0]:
import os
from pyspark.sql import functions as F

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, 'silver_output', 'parquet_data_hhs')
GOLD_PARQUET_DIR = os.path.join(BASE_DIR, 'gold_output', 'parquet_data_hhs')
os.makedirs(GOLD_PARQUET_DIR, exist_ok=True)

print(f"SILVER_PARQUET_DIR: {SILVER_PARQUET_DIR}")
print(f"GOLD_PARQUET_DIR:   {GOLD_PARQUET_DIR}")

In [0]:
# Read silver layer parquet tables

df_benefits = spark.read.parquet(
    os.path.join(SILVER_PARQUET_DIR, 'benefits_grouped_transformed')
)
df_rates = spark.read.parquet(
    os.path.join(SILVER_PARQUET_DIR, 'rate_baseline')
)

In [0]:
df_benefits_rates = df_rates.join(df_benefits, on="PlanId", how="inner")
display(df_benefits_rates)

In [0]:
df_benefits_rates.write.mode("overwrite").parquet(
    os.path.join(GOLD_PARQUET_DIR, 'benefits_rates')
)

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Exclude sentinel values, then cap at p99 for a readable distribution
df_clean = df_benefits_rates.filter(F.col("IndividualRate") < 999999)
p99 = df_clean.select(F.percentile_approx("IndividualRate", 0.99).alias("p99")).collect()[0]["p99"]
print(f"p99 cutoff: ${p99:,.2f}  — top 1% excluded from the visualisation")

pdf = (
    df_clean
    .filter(F.col("IndividualRate") <= p99)
    .select("IndividualRate", "BenefitCount")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pdf["BenefitCount"], pdf["IndividualRate"], alpha=0.35, color="steelblue", edgecolors="none", s=40)

# Trend line
z = np.polyfit(pdf["BenefitCount"], pdf["IndividualRate"], 1)
p = np.poly1d(z)
x_line = np.linspace(pdf["BenefitCount"].min(), pdf["BenefitCount"].max(), 200)
ax.plot(x_line, p(x_line), "r--", linewidth=2, label=f"Trend  (slope = {z[0]:.2f} $/benefit)")

ax.set_xlabel("BenefitCount", fontsize=12)
ax.set_ylabel("IndividualRate ($)", fontsize=12)
ax.set_title("IndividualRate vs BenefitCount — Scatter Plot", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

# Exclude sentinel values, then cap at p99 for a readable distribution
df_clean2 = df_benefits_rates.filter(F.col("IndividualRate") < 999999)
p99_2 = df_clean2.select(F.percentile_approx("IndividualRate", 0.99).alias("p99")).collect()[0]["p99"]

avg_rate_pdf = (
    df_clean2
    .filter(F.col("IndividualRate") <= p99_2)
    .groupBy("BenefitCount")
    .agg(
        F.avg("IndividualRate").alias("AvgIndividualRate"),
        F.count("*").alias("PlanCount")
    )
    .orderBy("BenefitCount")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(12, 6))

bars = ax1.bar(
    avg_rate_pdf["BenefitCount"],
    avg_rate_pdf["AvgIndividualRate"],
    color="steelblue", alpha=0.8, width=0.6, label="Avg IndividualRate"
)
ax1.set_xlabel("BenefitCount", fontsize=12)
ax1.set_ylabel("Avg IndividualRate ($)", fontsize=12, color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.set_title("Average IndividualRate by BenefitCount", fontsize=14)
ax1.set_xticks(avg_rate_pdf["BenefitCount"])

# Overlay plan count on a secondary axis
ax2 = ax1.twinx()
ax2.plot(
    avg_rate_pdf["BenefitCount"], avg_rate_pdf["PlanCount"],
    "r-o", linewidth=2, markersize=6, label="Plan Count"
)
ax2.set_ylabel("Number of Plans", fontsize=12, color="red")
ax2.tick_params(axis="y", labelcolor="red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc="upper left")

plt.tight_layout()
plt.show()

# Unit Tests